# Mobile Device Collector — SageMaker Runbook

One notebook to configure, install dependencies, and run the collector (same flow as `sagemaker_run.sh`).

## Before you start

1. Open this notebook from the **project root** folder (`device_maker_llm_scraper` / `mobile_device_collector`).
2. Use a **Python 3.11** kernel:
   - SageMaker Notebook Instance: **Kernel → Change kernel →** pick `device_collector` (create once in Terminal: `conda create -n device_collector python=3.11 -y`).
   - Or: **Kernel → Change kernel → conda_python3** only if it shows 3.11+.
3. Run cells **top to bottom** (Shift+Enter). **Always run Step 2** before dry-run or extraction (defines `run_main`).
4. Put your OpenAI API key only in **Cell 2 (Configuration)**. Do **not** commit the notebook after adding a real key.

## Where to check progress / errors

| What | Location |
|------|----------|
| **Main app log** (batch progress, errors, retries) | `data/output/run.log` |
| **Results** | `data/output/<input_name>_output.csv` and `.json` |
| **Resume checkpoint** | `data/output/checkpoint.json` |
| **This notebook** | Cell output below the run cell |

If the notebook shows "busy" or unknown status, open a **Terminal** and run:

```bash
tail -f data/output/run.log
```

Large runs (e.g. 26k devices) can take **many hours**. Keep the instance running; use **Resume** if interrupted.

## Step 1 — Configuration (edit this cell only)

In [ ]:
# ============ EDIT ONLY THIS CELL ============

OPENAI_API_KEY = ""  # Required: paste sk-proj-... here (do not commit)

INPUT_CSV = "data/input/all_device_model.csv"
BATCH_SIZE = 20
MAX_CONCURRENCY = 8
OUTPUT_DIR = "data/output"
LOG_LEVEL = "INFO"  # DEBUG | INFO | WARNING | ERROR

OPENAI_MODEL = "gpt-4o-mini"
ENABLE_CACHE = True

# Run options
RESUME = False          # True = continue from checkpoint.json
REPLAY_FAILED = False   # True = only retry failed batches
SKIP_DRY_RUN = False    # True = skip batch preview (not recommended)
INSTALL_DEPS = True     # False if dependencies already installed

# =============================================

## Step 2 — Apply settings & validate environment

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Project root = folder containing this notebook
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "app" / "main.py").exists():
    raise FileNotFoundError(
        f"app/main.py not found in {PROJECT_ROOT}. "
        "Open Jupyter from the project root (device_maker_llm_scraper)."
    )
os.chdir(PROJECT_ROOT)

if not OPENAI_API_KEY or OPENAI_API_KEY.startswith("sk-your-"):
    raise ValueError("Set OPENAI_API_KEY in the Configuration cell above.")

input_path = PROJECT_ROOT / INPUT_CSV
if not input_path.exists():
    raise FileNotFoundError(f"Input file not found: {input_path}")

py = sys.version_info
if py < (3, 11):
    raise RuntimeError(
        f"Python 3.11+ required; this kernel is {py.major}.{py.minor}.\n"
        "Fix: Terminal → conda create -n device_collector python=3.11 -y\n"
        "Then: Kernel → Change kernel → device_collector"
    )

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["OPENAI_MODEL"] = OPENAI_MODEL
os.environ["BATCH_SIZE"] = str(BATCH_SIZE)
os.environ["MAX_CONCURRENCY"] = str(MAX_CONCURRENCY)
os.environ["LOG_LEVEL"] = LOG_LEVEL
os.environ["ENABLE_CACHE"] = "true" if ENABLE_CACHE else "false"

LOG_FILE = PROJECT_ROOT / OUTPUT_DIR / "run.log"
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Python       :", sys.version.split()[0], "@", sys.executable)
print("Input CSV    :", input_path)
print("Output dir   :", PROJECT_ROOT / OUTPUT_DIR)
print("App log file :", LOG_FILE)
print("API key      :", OPENAI_API_KEY[:12] + "...")


def run_main(extra_args: list[str]) -> int:
    """Run app/main.py as a subprocess; stream output to the notebook."""
    cmd = [
        sys.executable,
        "app/main.py",
        "--input",
        str(input_path),
        "--batch-size",
        str(BATCH_SIZE),
        "--concurrency",
        str(MAX_CONCURRENCY),
        "--output-dir",
        OUTPUT_DIR,
        "--log-level",
        LOG_LEVEL,
    ] + extra_args
    print("$", " ".join(cmd))
    print("(App logs also written to:", LOG_FILE, ")\n")
    proc = subprocess.Popen(
        cmd,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}")
    return proc.returncode


print("run_main() ready — use in dry-run and extraction cells below.")

## Step 3 — Install dependencies

Uses pre-built wheels for NumPy/Pandas and `requirements-sagemaker.txt` (no tiktoken compile).
Tiktoken is installed from a wheel or conda if possible; **the app still runs without it**.
Safe to re-run.

In [ ]:
import shutil
import subprocess

if not INSTALL_DEPS:
    print("INSTALL_DEPS=False — skipping pip install.")
else:

    def pip(*args, check=True):
        cmd = [sys.executable, "-m", "pip"] + list(args)
        print("$", " ".join(cmd))
        r = subprocess.run(cmd, cwd=PROJECT_ROOT, check=False)
        if check and r.returncode != 0:
            raise subprocess.CalledProcessError(r.returncode, cmd)
        return r

    def install_tiktoken_optional():
        """Avoid Rust/GCC source builds on old SageMaker AMIs."""
        r = pip("install", "tiktoken>=0.7.0", "--only-binary=:all:", check=False)
        if r.returncode == 0:
            import tiktoken

            print("tiktoken", tiktoken.__version__, "(pip wheel)")
            return
        conda = shutil.which("conda")
        if conda:
            print("$", conda, "install -y -c conda-forge tiktoken")
            r2 = subprocess.run(
                [conda, "install", "-y", "-c", "conda-forge", "tiktoken"],
                check=False,
            )
            if r2.returncode == 0:
                import tiktoken

                print("tiktoken", tiktoken.__version__, "(conda-forge)")
                return
        print(
            "WARNING: tiktoken not installed — using character-based token estimates. "
            "This is OK for production runs."
        )

    pip("install", "--upgrade", "pip", "wheel")
    pip("install", "numpy>=1.26.0,<2.0.0", "pandas>=2.2.0,<3.0.0")
    req = PROJECT_ROOT / "requirements-sagemaker.txt"
    if not req.exists():
        raise FileNotFoundError(f"Missing {req} — pull latest code from git.")
    pip("install", "-r", str(req))
    install_tiktoken_optional()

    import numpy as np
    import pandas as pd
    import openai

    print("OK — numpy", np.__version__, "| pandas", pd.__version__, "| openai", openai.__version__)

## Step 4 — Dry run (batch plan, no API calls)

In [ ]:
if SKIP_DRY_RUN:
    print("SKIP_DRY_RUN=True — skipping dry run.")
else:
    run_main(["--dry-run"])

## Step 5 — Run extraction

**This cell may run for hours** on large CSV files. Progress is logged to `data/output/run.log` even if this cell looks stuck.

To monitor in another Terminal: `tail -f data/output/run.log`

In [ ]:
import time

extra = []
if RESUME:
    extra.append("--resume")
if REPLAY_FAILED:
    extra.append("--replay-failed")

print("=" * 60)
print("  MOBILE DEVICE COLLECTOR — STARTING")
print("=" * 60)
print(f"  Input        : {INPUT_CSV}")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  Concurrency  : {MAX_CONCURRENCY}")
print(f"  Resume       : {RESUME}")
print(f"  Replay failed: {REPLAY_FAILED}")
print(f"  Log file     : {LOG_FILE}")
print("=" * 60)

if not SKIP_DRY_RUN:
    print("Starting in 3 seconds... (Interrupt kernel to cancel)")
    time.sleep(3)

run_main(extra)

out_stem = input_path.stem
print("\nDone. Outputs:")
for p in sorted((PROJECT_ROOT / OUTPUT_DIR).glob(f"{out_stem}_output.*")):
    print(" ", p)

## Step 6 — View recent log lines

Run anytime (while job is running or after) to inspect `data/output/run.log`.

In [ ]:
LOG_TAIL_LINES = 80

if not LOG_FILE.exists():
    print(f"No log file yet: {LOG_FILE}")
else:
    lines = LOG_FILE.read_text(encoding="utf-8", errors="replace").splitlines()
    print(f"--- Last {min(LOG_TAIL_LINES, len(lines))} lines of {LOG_FILE} ---\n")
    print("\n".join(lines[-LOG_TAIL_LINES:]))